# NB1 — Player Pool

Goal: turn the frozen FBref snapshot into one clean, tiered, role-labelled player pool that
NB2/NB3 and `scoring.py` can consume without further cleaning.

Output contract (what downstream code relies on):
- one row per `(player, season)` — key `Url, Season_End_Year, Squad`
- every metric column referenced by `config/mourinho_weights.json`, numeric
- `role` (one of the 6 archetype roles), `Min_Playing` (primary-club minutes, the per-90 denominator),
  `season_total_min` (minutes summed across clubs, used only for the tier filter), `tier` (A/B/C/None)

Prerequisite: run `bootstrap_data.py` first so `data/raw/fbref/*.parquet` exist.

Design choices, stated up front:
- Split seasons (a player at 2 clubs in one season) → keep the higher-minutes club row for stats,
  but tier on summed minutes. ~4% of player-seasons. Simple and defensible; full stat-aggregation is a later refinement.
- Role* default to a FBref-position heuristic (runs standalone). True CB-vs-FB and DM resolution needs
  Transfermarkt `sub_position` (see the optional enrichment cell).
- Per-90 conversion lives in `scoring.py`, not here. NB1 emits raw counts + minutes.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# --- paths (run from notebooks/) ---
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "fbref"
OUT = ROOT / "data" / "transformed"; OUT.mkdir(parents=True, exist_ok=True)
WEIGHTS = ROOT / "config" / "mourinho_weights.json"

SEASONS = [2021, 2022, 2023]          # 2020-21, 2021-22, 2022-23
LATEST  = max(SEASONS)

# Tier thresholds (edit here)
TIER_A_MIN, TIER_A_NSEAS = 1500, 2     # >=1500 min in >=2 seasons
TIER_B_MIN               = 900         # >=900 min in latest season + upward trend
TIER_C_MIN, TIER_C_NSEAS, TIER_C_AGE = 500, 2, 21

## Step 1 — Load the raw tables

In [ ]:
TABLES = ["standard","shooting","passing","defense","possession","misc","gca","playing_time"]
missing = [t for t in TABLES if not (RAW / f"{t}.parquet").exists()]
# only strictly need the tables referenced below; warn if the core ones are absent
core = ["standard","passing","defense","possession","misc","gca"]
absent = [t for t in core if not (RAW / f"{t}.parquet").exists()]
assert not absent, f"Missing parquet for {absent}. Run bootstrap_data.py first."

raw = {t: pd.read_parquet(RAW / f"{t}.parquet") for t in core}
for t, df in raw.items():
    print(f"{t:11s} {df.shape}")

standard    (8084, 34)
passing     (8060, 34)
defense     (8060, 33)
possession  (8060, 36)
misc        (8060, 26)
gca         (8060, 26)


## Step 2 — Derive the column contract from the weights file, then merge

Read which columns each table must contribute straight from `mourinho_weights.json`,
so if you add a metric to the weights later, NB1 picks it up automatically.

In [6]:
KEY = ["Url", "Season_End_Year", "Squad"]
IDENTITY = ["Player", "Comp", "Pos", "Age", "Min_Playing"]   # taken from the standard spine

# Build {table: [columns]} from the weights file (falls back to a hardcoded contract).
def contract_from_weights(path):
    need = {}
    spec = json.load(open(path))
    for role in spec["roles"].values():
        for m in role["metrics"].values():
            need.setdefault(m["table"], set()).add(m["column"])
    return {t: sorted(c) for t, c in need.items()}

try:
    NEED = contract_from_weights(WEIGHTS)
    print("Contract loaded from weights file.")
except FileNotFoundError:
    NEED = {  # safety fallback — mirrors the v1 weights file
        "standard":  ["G_minus_PK_Per","npxG_Per","xAG_Per"],
        "passing":   ["Cmp_percent_Total","CrsPA","Final_Third","KP","Prog"],
        "defense":   ["Blocks_Blocks","Clr","Err","Int","Mid 3rd_Tackles","Tkl+Int","TklW_Tackles"],
        "possession":["Att Pen_Touches","Final_Third_Carries","Mis_Carries","Prog_Carries","Prog_Receiving","Succ_Dribbles"],
        "misc":      ["Recov","Won_Aerial","Won_percent_Aerial"],
        "gca":       ["GCA90_GCA","SCA90_SCA"],
    }
    print("Weights file not found — using fallback contract.")

# Spine = standard (carries identity + minutes + its own contract cols)
std_cols = KEY + IDENTITY + [c for c in NEED.get("standard", []) if c not in IDENTITY]
pool = raw["standard"][std_cols].copy()

# Select-then-merge: only the key + contract columns from each other table (no _x/_y collisions)
for t in ["passing","defense","possession","misc","gca"]:
    cols = KEY + NEED.get(t, [])
    pool = pool.merge(raw[t][cols], on=KEY, how="left", validate="one_to_one")

assert len(pool) == len(raw["standard"]), "Row count changed on merge — investigate the key."
print("merged pool:", pool.shape)
pool.head(3)

Contract loaded from weights file.
merged pool: (8084, 34)


,Url,Season_End_Year,Squad,Player,Comp,Pos,Age,Min_Playing,G_minus_PK_Per,npxG_Per,...,Final_Third_Carries,Mis_Carries,Prog_Carries,Prog_Receiving,Succ_Dribbles,Recov,Won_Aerial,Won_percent_Aerial,GCA90_GCA,SCA90_SCA
0,https://fbref.com/en/players/355c883a/Martin-A...,2021,Alavés,Martin Agirregabiria,La Liga,"DF,MF",24,1558.0,0.00,0.01,...,8.0,8.0,33.0,19.0,4.0,126.0,25.0,51.0,0.17,1.33
1,https://fbref.com/en/players/d637fc22/Rodrigo-...,2021,Alavés,Rodrigo Battaglia,La Liga,MF,29,2492.0,0.04,0.04,...,23.0,15.0,78.0,16.0,18.0,259.0,45.0,65.2,0.04,0.65
2,https://fbref.com/en/players/45e304c8/Burgui,2021,Alavés,Burgui,La Liga,MF,26,21.0,0.00,0.00,...,2.0,0.0,3.0,1.0,0.0,2.0,1.0,100.0,0.00,0.00


## Step 3 — Clean: age → int, primary position, drop keepers / blanks

In [7]:
# numeric coercion for every contract metric (snapshot stores some as strings)
metric_cols = [c for cols in NEED.values() for c in cols]
for c in set(metric_cols + ["Min_Playing"]):
    pool[c] = pd.to_numeric(pool[c], errors="coerce")

# Age format is INCONSISTENT across seasons: some "24", some "33-113" (years-days).
# Take the years part before any dash, then coerce.
pool["Age"] = (pool["Age"].astype(str).str.split("-").str[0]
               .pipe(pd.to_numeric, errors="coerce").astype("Int64"))

# primary position = first listed (FBref encodes priority by order)
pool["Pos"] = pool["Pos"].astype(str).str.strip()
pool["prim_pos"] = pool["Pos"].str.split(",").str[0].str.strip()

before = len(pool)
pool = pool[~pool["prim_pos"].isin(["GK", "", "nan"])].copy()      # drop keepers + blanks
print(f"dropped {before - len(pool)} GK/blank rows; remaining {len(pool)}")

dropped 570 GK/blank rows; remaining 7514


## Step 4 — Split seasons

No combined row exists, so for each `(Url, Season_End_Year)` we sum minutes across clubs for the
tier filter and keep the higher-minutes club's row for the stats (its per-90s are a real sample).

In [8]:
grp = pool.groupby(["Url","Season_End_Year"])
season_min = grp["Min_Playing"].transform("sum")
n_clubs    = grp["Url"].transform("size")
pool["season_total_min"] = season_min
pool["n_clubs"] = n_clubs

# keep the max-minutes row per (player, season)
idx = grp["Min_Playing"].idxmax()
pool = pool.loc[idx].reset_index(drop=True)
print(f"split-season players collapsed; one row per (player, season): {len(pool)} rows")
print("players who split a season:", int((pool['n_clubs'] > 1).sum()))

split-season players collapsed; one row per (player, season): 7222 rows
players who split a season: 290


## Step 5 — Role assignment

Default heuristic from FBref position (runs standalone). It populates 5 of the 6 roles; **defensive_mid
cannot be separated from central midfield in FBref data** and is only resolved by the optional
Transfermarkt enrichment below.

In [9]:
def role_from_pos(pos):
    parts = [p.strip() for p in str(pos).split(",")]
    prim = parts[0]; sec = parts[1] if len(parts) > 1 else None
    if prim == "DF": return "full_back" if sec in ("MF","FW") else "center_back"
    if prim == "MF": return "wide_attacker" if sec == "FW" else "central_mid"
    if prim == "FW": return "wide_attacker" if sec == "MF" else "striker"
    return None

pool["role"] = pool["Pos"].apply(role_from_pos)
pool = pool[pool["role"].notna()].copy()
print(pool["role"].value_counts())
print("\nNote: no 'defensive_mid' yet — enrich from Transfermarkt to unlock it.")

role
center_back      2398
central_mid      2253
wide_attacker    1337
striker           827
full_back         407
Name: count, dtype: int64

Note: no 'defensive_mid' yet — enrich from Transfermarkt to unlock it.


In [10]:
# --- OPTIONAL: upgrade roles with Transfermarkt sub_position --------------------
# Wire this once you've pulled dcaribou in NB2. Pass a df with player name + sub_position.
import unicodedata
def _norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode().lower().strip()
    return " ".join(s.split())

def enrich_roles(pool, tm_players, name_col="name", subpos_col="sub_position", weights_path=WEIGHTS):
    pmap = json.load(open(weights_path)).get("_position_map", {})
    tm = tm_players[[name_col, subpos_col]].dropna().copy()
    tm["_k"] = tm[name_col].map(_norm)
    tm["_role"] = tm[subpos_col].map(pmap)
    tm = tm.dropna(subset=["_role"]).drop_duplicates("_k").set_index("_k")["_role"]
    pool = pool.copy()
    pool["_k"] = pool["Player"].map(_norm)
    matched = pool["_k"].isin(tm.index)
    pool.loc[matched, "role"] = pool.loc[matched, "_k"].map(tm)
    print(f"TM role match rate: {matched.mean():.1%} ({matched.sum()}/{len(pool)})")
    return pool.drop(columns="_k")

# Example (uncomment when ready):
# tm_players = pd.read_csv(ROOT/'data'/'raw'/'transfers'/'players.csv')
# pool = enrich_roles(pool, tm_players)

## Step 6 — Tiering

Per the rules: assess each player across the seasons they appear. Tier is decided at the player level,
then mapped back onto their season rows.

In [11]:
# per-(player) season minutes and ages
by_player = pool.groupby("Url")
season_tbl = (pool.pivot_table(index="Url", columns="Season_End_Year",
                               values="season_total_min", aggfunc="max"))

def tier_for(url):
    mins = season_tbl.loc[url].dropna()
    if (mins >= TIER_A_MIN).sum() >= TIER_A_NSEAS:
        return "A"
    latest_min = mins.get(LATEST, np.nan)
    upward = (len(mins) >= 2 and pd.notna(latest_min)
              and latest_min == mins.max() and latest_min > mins.iloc[0])
    if pd.notna(latest_min) and latest_min >= TIER_B_MIN and upward:
        return "B"
    sub = pool[pool["Url"] == url]
    latest_age = sub.loc[sub["Season_End_Year"] == LATEST, "Age"]
    young = (not latest_age.empty) and pd.notna(latest_age.iloc[0]) and (latest_age.iloc[0] <= TIER_C_AGE)
    if bool(young) and (mins >= TIER_C_MIN).sum() >= TIER_C_NSEAS:
        return "C"
    return None

tier_map = {u: tier_for(u) for u in season_tbl.index}
pool["tier"] = pool["Url"].map(tier_map)
print(pool.drop_duplicates("Url")["tier"].value_counts(dropna=False))

tier
NaN    3012
A       641
C        62
B        23
Name: count, dtype: int64


## Step 7 — Save outputs

In [12]:
pool.to_csv(OUT / "fbref_merged.csv", index=False)
for t in ["A","B","C"]:
    sub = pool[pool["tier"] == t]
    sub.to_csv(OUT / f"tier_{t}_players.csv", index=False)
    print(f"tier_{t}: {sub['Url'].nunique()} players, {len(sub)} season-rows")

print("\nSaved to", OUT)
print("Downstream (scoring.py) reads tier_A_players.csv, aggregates per-90 across seasons, "
      "percentiles within role, applies mourinho_weights.json.")

tier_A: 641 players, 1856 season-rows
tier_B: 23 players, 53 season-rows
tier_C: 62 players, 168 season-rows

Saved to c:\Users\Gilbert\Documents\Projects\real-madrid-scouting\data\transformed
Downstream (scoring.py) reads tier_A_players.csv, aggregates per-90 across seasons, percentiles within role, applies mourinho_weights.json.
